<a href="https://colab.research.google.com/github/nenurd44/ASR-model-comparison/blob/main/comparing_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate datasets soundfile librosa faster-whisper jiwer

In [ ]:
import pandas as pd
from jiwer import wer, cer
import time
import re

audios = [
    {
        "file": "test_en_1.wav",
        "text": "okay let's go let's start our testing",
        "lang": "en"
    },
    {
        "file": "test_en_2.wav",
        "text": "now im going to the university so i can work on my project",
        "lang": "en"
    },
    {
        "file": "test_en_3.wav",
        "text": "the weather today is sunny and very hot to be honest",
        "lang": "en"
    },
    {
        "file": "test_en_4.wav",
        "text": "im really tired i just wanna go home and sleep",
        "lang": "en"
    },
    {
        "file": "test_en_5.wav",
        "text": "hello what do you think who is the best singer alive",
        "lang": "en"
    },
    {
        "file": "test_en_6.wav",
        "text": "hey siri how to find the purpose of life",
        "lang": "en"
    },
    # ---------- ru
    {
        "file": "test_ru_1.wav",
        "text": "всем привет сегодня у нас погода отличная",
        "lang": "ru"
    },
    {
        "file": "test_ru_2.wav",
        "text": "сегодня у нас наурыз и народа тут очень много",
        "lang": "ru"
    },
    {
        "file": "test_ru_3.wav",
        "text": "когда мы покупали впервые машину там такой приятный запах",
        "lang": "ru"
    },
    {
        "file": "test_ru_4.wav",
        "text": "я люблю гулять ночью потому что ночью обычно тихо на улице",
        "lang": "ru"
    },
    {
        "file": "test_ru_5.wav",
        "text": "привет алиса поставь пожалуйста будильник на семь утра",
        "lang": "ru"
    },
    {
        "file": "test_ru_6.wav",
        "text": "привет а какова вероятность что лифт упадет",
        "lang": "ru"
    },
    # ---------- mix
    {
        "file": "test_mix_1.wav",
        "text": "вчера у нас была презентация i guess it was cringe",
        "lang": "mix"
    },
    {
        "file": "test_mix_2.wav",
        "text": "we planned to go to swimming но там было занято",
        "lang": "mix"
    },
    {
        "file": "test_mix_3.wav",
        "text": "я смотрел the first spider man и мне это очень сильно понравилось",
        "lang": "mix"
    },
    {
        "file": "test_mix_4.wav",
        "text": "hello hello what will be the погода завтра",
        "lang": "mix"
    },
    {
        "file": "test_mix_5.wav",
        "text": "это мне кажется или one plus one is really three",
        "lang": "mix"
    },
    {
        "file": "test_mix_6.wav",
        "text": "с днём рождения тебя i wish you all the best",
        "lang": "mix"
    },
]

results = []

def basic_clean(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())

def record_result(model_name, audio, pred, duration):
    audio["text"] = basic_clean(audio["text"])
    prediction = basic_clean(pred)

    results.append({
        "Model": model_name,
        "File": audio["file"],
        "Language": audio["lang"],
        "Real": audio["text"],
        "Predicted": prediction.strip(),
        "WER": round(wer(audio["text"], prediction), 4),
        "CER": round(cer(audio["text"], prediction), 4),
        "Time_Sec": round(duration, 2)
    })

In [ ]:
import torch
from transformers import pipeline

model1 = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch.float16,
    device="cuda"
)

for audio in audios:
    start = time.time()
    output = model1(audio["file"])
    record_result("Whisper-v3", audio, output["text"], time.time() - start)

del model1
torch.cuda.empty_cache()

In [ ]:
from faster_whisper import WhisperModel
import time

model2 = WhisperModel("large-v3", device="cuda", compute_type="float16")

for audio in audios:
    start = time.time()

    segments, info = model2.transcribe(audio["file"], beam_size=1)

    pred = "".join([segment.text for segment in segments])

    record_result("Faster-Whisper-V3", audio, pred, time.time() - start)

del model2
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import pipeline

model3 = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16,
    device="cuda"
)

for audio in audios:
    start = time.time()
    result = model3(audio["file"])
    record_result("Whisper-Turbo", audio, result["text"], time.time() - start)

del model3
torch.cuda.empty_cache()

dalwe rezy

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame(results)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
plot = sns.barplot(data=df, x="Model", y="WER", hue="Language", palette="viridis")
plt.title("ASR Performance: Word Error Rate (Lower is Better)")
plt.ylabel("WER (0.0 - 1.0)")
plt.show()

df_sorted = df.sort_values(by=["WER", "CER", "Time_Sec"], ascending=[True, True, True])

display(df_sorted[["Model", "Language", "WER", "CER", "Time_Sec", "Predicted"]].reset_index(drop=True))

In [ ]:
plt.savefig("wer_by_model.png")
plt.savefig("wer_by_language.png")

In [ ]:
import pandas as pd

model_summary = (
    df.groupby("Model")
      .agg({
          "WER": "mean",
          "CER": "mean",
          "Time_Sec": "mean"
      })
      .reset_index()
      .sort_values("WER")
)

print(model_summary)

In [ ]:
wer_lang = (
    df.groupby(["Model", "Language"])["WER"]
      .mean()
      .reset_index()
)

wer_pivot = wer_lang.pivot(index="Model", columns="Language", values="WER")

print(wer_pivot)

In [ ]:
latency = (
    df.groupby("Model")["Time_Sec"]
      .mean()
      .reset_index()
      .sort_values("Time_Sec")
)

print(latency)

In [ ]:
comparison = (
    df.groupby("Model")["WER"]
      .describe()
)

print(comparison)

In [ ]:
hardest = df.sort_values("WER", ascending=False).head(10)

print(hardest[["Model", "Language", "WER", "Predicted"]])

In [ ]:
model_summary.to_csv("model_summary.csv", index=False)
wer_pivot.to_csv("wer_by_language.csv")
latency.to_csv("latency.csv", index=False)

trash dalwe

In [ ]:
# !pip install jiwer
# from jiwer import wer, cer

In [ ]:
# org = "hello my name is baurzhan im from kazakhstan"
# pred = "hello my name is pao zhang im from kazakhstan"
# print(round(wer(org, pred), 4))

In [ ]:
# from nemo.collections.asr.models import EncDecMultiTaskModel

# canary_model = EncDecMultiTaskModel.from_pretrained('nvidia/canary-1b')
# canary_model.to("cuda")

# for audio in audios:
#     start_time = time.time()
#     res = canary_model.transcribe(
#         paths2audio_files=[audio["file"]],
#         task="asr",
#         source_lang=audio["lang"],
#         batch_size=1
#     )

#     prediction = res[0] if isinstance(res, list) else res

#     record_result("Canary-1B", audio, prediction, time.time() - start_time)

# del canary_model
# torch.cuda.empty_cache()
